In [36]:
%pip install konlpy

/bin/bash: /home/medinfo/anaconda3/envs/sbert/lib/libtinfo.so.6: no version information available (required by /bin/bash)

[notice] A new release of pip is available: 23.3.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
import os
from konlpy.tag import Komoran
import pandas as pd
import re

In [38]:
LEX_PATH = './lexicon'
LEX_FILES = os.listdir(LEX_PATH)


In [39]:
filepath = './lexicon/polarity.csv'
lexicon = pd.read_csv(filepath)
lexicon['ngram'] = lexicon['ngram'].str.replace(';', ' ')
lexicon['COMP'] = (lexicon['COMP'] * lexicon['freq']).astype('int')
lexicon['NEG'] = (lexicon['NEG'] * lexicon['freq']).astype('int')
lexicon['NEUT'] = (lexicon['NEUT'] * lexicon['freq']).astype('int')
lexicon['None'] = (lexicon['None'] * lexicon['freq']).astype('int')
lexicon['POS'] = (lexicon['POS'] * lexicon['freq']).astype('int')
lexicon['n'] = lexicon['ngram'].str.count(' ') + 1
lexicon = lexicon[~lexicon['ngram'].str.contains('*/', regex=False)]
lexicon = lexicon.sort_values('max.prop', ascending=False)
lexicon = lexicon.sort_values('n', ascending=False)
# lexicon['ngram'] = lexicon['ngram'].str.replace('/[A-Z]+', '', regex=True)
lexicon.set_index('ngram', inplace=True)
lexicon

,freq,COMP,NEG,NEUT,None,POS,max.value,max.prop,n
ngram,,,,,,,,,
세대/NNG 바람/NNG 이/JKS,1,0,0,0,0,1,POS,1.0,3
방불/XR 하/XSA 게/EC,2,0,0,0,0,2,POS,1.0,3
한/MM 방법/NNG 이/VCP,1,0,0,0,0,1,POS,1.0,3
가능/NNG 하/XSA ㄴ/ETM,1,0,0,0,0,1,POS,1.0,3
한/MM 목소리/NNG 를/JKO,1,0,0,0,0,1,POS,1.0,3
...,...,...,...,...,...,...,...,...,...
살아나/VV,1,0,0,0,0,1,POS,1.0,1
살아남/VV,1,0,0,0,0,1,POS,1.0,1
살아오/VV,1,0,0,0,0,1,POS,1.0,1


In [40]:
min_freq = 5
threshold = 0.5
selected = lexicon[(lexicon['freq'] >= min_freq) & (lexicon['max.prop'] > threshold)].index.tolist()
selected

['하/VV 지/EC 않/VX',
 '하/XSA 지/EC 않/VX',
 '을/ETM 수/NNB 없/VA',
 '하/XSV 아/EC 주/VX',
 'ㄹ/ETM 수/NNB 없/VA',
 '하/XSV ㄹ/ETM 수/NNB',
 '지/EC 않/VX 으면/EC',
 '을/ETM 수/NNB 있/VV',
 '기/ETN 도/JX 하/VV',
 '하/XSV 지/EC 못하/VX',
 '가장/MAG 크/VA ㄴ/ETM',
 '이/JKC 아니/VCN 라/EC',
 '을/ETM 것/NNB 이/VCP',
 '기/ETN 도/JX 하/VX',
 '수/NNB 밖에/JX 없/VA',
 'ㄹ/ETM 수/NNB 밖에/JX',
 '어/EC 있/VX 는/ETM',
 'ㄹ/ETM 예정/NNG 이/VCP',
 '는/ETM 것/NNB 이/VCP',
 '수/NNB 도/JX 있/VV',
 'ㄴ/ETM 것/NNB 이/VCP',
 'ㄹ/ETM 것/NNB 이/VCP',
 'ㄹ/ETM 수/NNB 도/JX',
 '알리/VV 어/EC 지/VX',
 '것/NNB 이/VCP 다/EF',
 '적/XSN 이/VCP ㄴ/ETM',
 'ㄹ/ETM 수/NNB 있/VV',
 '는/ETM 것/NNB 이/JKS',
 '기/ETN 마련/NNB 이/VCP',
 '하/XSA 아/EC 지/VX',
 'ㄹ/ETM 만/NNB 하/XSA',
 'ㄴ/ETM 것/NNB 이/JKS',
 '느끼/VV 어/EC 지/VX',
 '눈/NNG 에/JKB 띄/VV',
 '말/NNG 하/XSV 아/EC',
 '적/VA 지/EC 않/VX',
 '기/ETN 로/JKB',
 '적/VA 지/EC',
 '아이/NNG 들/XSN',
 '가/JKS 많/VA',
 '도/JX 있/VV',
 '수/NNB 있/VV',
 'ㄴ/ETM 일/NNG',
 '어/EC 주/VX',
 '화/XSN 되/XSV',
 '하/VX 는/ETM',
 '이/JKS 있/VV',
 '하/VX ㄹ/ETM',
 '가/JKC 아니/VCN',
 '눈/NNG 에/JKB',
 '알리/VV 어/EC',
 '수/NNB 도/JX

In [41]:
unigrams = lexicon[lexicon['n'] == 1].index.tolist()
DICT_PATH = './tagger/user_dictionary.txt'
with open(DICT_PATH, 'w') as f:
    f.writelines('\n'.join(['\t'.join(unigram.split('/')) for unigram in unigrams]))

In [42]:
pattern = re.compile('|'.join(selected))
pattern


re.compile(r'하/VV 지/EC 않/VX|하/XSA 지/EC 않/VX|을/ETM 수/NNB 없/VA|하/XSV 아/EC 주/VX|ㄹ/ETM 수/NNB 없/VA|하/XSV ㄹ/ETM 수/NNB|지/EC 않/VX 으면/EC|을/ETM 수/NNB 있/VV|기/ETN 도/JX 하/VV|하/XSV 지/EC 못하/VX|가장/MAG 크/VA ㄴ/ETM|이/JKC 아니/VCN 라/EC|을/ETM 것/NNB 이/VCP|기/ETN 도/JX 하/VX|수/NNB 밖에/JX 없/VA|ㄹ/ETM 수/NNB 밖에/JX|어/EC 있/VX 는/ETM|ㄹ/ETM 예정/NNG 이/VCP|는/ETM 것/NNB 이/VCP|수/NNB 도/JX 있/VV|ㄴ/ETM 것/NNB 이/VCP|ㄹ/ETM 것/NNB 이/VCP|ㄹ/ETM 수/NNB 도/JX|알리/VV 어/EC 지/VX|것/NNB 이/VCP 다/EF|적/XSN 이/VCP ㄴ/ETM|ㄹ/ETM 수/NNB 있/VV|는/ETM 것/NNB 이/JKS|기/ETN 마련/NNB 이/VCP|하/XSA 아/EC 지/VX|ㄹ/ETM 만/NNB 하/XSA|ㄴ/ETM 것/NNB 이/JKS|느끼/VV 어/EC 지/VX|눈/NNG 에/JKB 띄/VV|말/NNG 하/XSV 아/EC|적/VA 지/EC 않/VX|기/ETN 로/JKB|적/VA 지/EC|아이/NNG 들/XSN|가/JKS 많/VA|도/JX 있/VV|수/NNB 있/VV|ㄴ/ETM 일/NNG|어/EC 주/VX|화/XSN 되/XSV|하/VX 는/ETM|이/JKS 있/VV|하/VX ㄹ/ETM|가/JKC 아니/VCN|눈/NNG 에/JKB|알리/VV 어/EC|수/NNB 도/JX|이/VCP ㅂ니다/EF|과대/NNG 포장/NNG|하/VV 았/EP|고/EC 싶/VX|하/XSV 지/EC|가/JKS 없/VA|어/EC 지/VX|을/ETM 것/NNB|가/JKS 있/VV|적/XSN 이/VCP|지/EC 않/VX|었/EP 습니다/EF|되/XSV 지/EC|설명/NNG 하/XSV|주장/NNG 하/XSV|하/XSV ㄴ/ETM|크/VA ㄴ/

In [48]:
# sentence = '모든 인간은 태어날 때부터 자유로우며 그 존엄과 권리에 있어 동등하다. 인간은 천부적으로 이성과 양심을 부여받았으며 서로 형제애의 정신으로 행동하여야 한다.'
sentence = '너무 고통스러워서 차마 볼 수 없었습니다.'
sentence = '정말 행복합니다.'

In [49]:
tagger = Komoran(userdic=DICT_PATH)

In [50]:
tagger.pos(sentence)
tagged = ' '.join(tagger.pos(sentence, join=True))
tagged

'정말/MAG 행복/NNG 하/XSV ㅂ니다/EF ./SF'

In [51]:
matches = pattern.findall(tagged)
result = [(match, lexicon.loc[match]['max.value'], lexicon.loc[match]['max.prop']) for match in matches]
result

[('행복/NNG', 'POS', 0.666666667),
 ('하/XSV', 'POS', 0.516129032),
 ('ㅂ니다/EF', 'POS', 0.727272727)]

In [52]:
# naive bayes 방식으로 긍정/부정 가능도 계산하기

0 -1.3853173218470638
